# Classical TF-IDF baseline · Kaggle execution

Upload this notebook as a **private** Kaggle notebook. Attach the competition dataset and a private dataset containing the code and frozen-split ZIP contents. Enable Internet for installing the pinned runtime dependencies. GPU T4 x2 is compatible, but this scikit-learn experiment executes on CPU; no GPU acceleration or model change is introduced.

**Run All trains by default:** four candidates × three CV folds on the same frozen 30,002-record sample, followed by the CV-selected model fitted on all 217,881 development records and evaluated on the 54,471-record holdout. The setup/audit cells do not train. To inspect setup only, set `RUN_TRAINING = False` below. Optional all-data refit/test export is disabled by default. There is no automatic Kaggle submission.

In [ ]:
from pathlib import Path
import csv
import hashlib
import html
import json
import logging
import subprocess
import sys
import tomllib

from IPython.display import HTML, FileLink, display

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/hm3_baseline")
TEMPORARY_ROOT = Path("/kaggle/temp/hm3_baseline")
RUN_TRAINING = True
RUN_FINAL_SUBMISSION = False
FINAL_STAGE = "submission"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger("kaggle_baseline")

## 01 · Verify uploaded source before importing it

Kaggle unpacks dataset ZIP uploads. The bundle marker identifies the code without assuming your private dataset slug. Every source file is checked against its bundle checksum before imports. Exactly one source version and one frozen-split version must be attached.

In [ ]:
code_markers = [
    path
    for path in INPUT_ROOT.rglob("hm3_bundle.json")
    if json.loads(path.read_text()).get("bundle_kind") == "code"
]
if len(code_markers) != 1:
    raise ValueError(
        "Attach one private code bundle; upload its ZIP as a Kaggle Dataset so it is unpacked"
    )
CODE_ROOT = code_markers[0].parent
code_record = json.loads(code_markers[0].read_text())
for name, expected in code_record["files"].items():
    path = (CODE_ROOT / name).resolve()
    if not path.is_relative_to(CODE_ROOT.resolve()):
        raise ValueError("Invalid bundle path")
    with path.open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    if actual != expected:
        raise ValueError(f"Uploaded code checksum mismatch: {name}")
sys.path.insert(0, str(CODE_ROOT / "src"))
from hm3.kaggle_bootstrap import find_bundle, prepare_runtime

SPLIT_ROOT = find_bundle(INPUT_ROOT, "split")
config = tomllib.loads((CODE_ROOT / "config/baseline_kaggle.toml").read_text())
display({"code": str(CODE_ROOT), "split": str(SPLIT_ROOT), "configuration": config})

## 02 · Isolated environment and source audit

The isolated environment uses the hosted interpreter when it is Python 3.12–3.14. It installs only hash-pinned CPU dependencies using uv; the kernel's CUDA/PyTorch stack is untouched. A missing uv executable is obtained from its official release with a pinned archive checksum. Python packages and setup tools require Internet; no external text data is downloaded.

The audit verifies the actual attached train fingerprint, restores the exact frozen IDs and checks duplicate/fold boundaries. It records Python/library versions, hardware, disk capacity, recipe identity and split distributions. Metadata is copied to saved outputs for private restoration. No split regeneration is permitted on Kaggle.

In [ ]:
RUNTIME_PYTHON = prepare_runtime(CODE_ROOT, TEMPORARY_ROOT)
ENTRY_POINT = CODE_ROOT / "scripts/run_kaggle_baseline.py"


def run_stage(stage: str) -> None:
    command = [
        str(RUNTIME_PYTHON),
        str(ENTRY_POINT),
        "--code-root",
        str(CODE_ROOT),
        "--split-root",
        str(SPLIT_ROOT),
        "--input-root",
        str(INPUT_ROOT),
        "--output-root",
        str(OUTPUT_ROOT),
        "--stage",
        stage,
    ]
    with subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as process:
        if process.stdout is None:
            raise RuntimeError("Runner stdout is unavailable")
        for line in process.stdout:
            logger.info("%s", line.rstrip())
        return_code = process.wait()
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)


run_stage("audit")
pointer = json.loads((OUTPUT_ROOT / "current_run.json").read_text())
RUN_PATH = OUTPUT_ROOT / pointer["run_directory"]
run_metadata = json.loads((RUN_PATH / "run_metadata.json").read_text())
audit = json.loads((RUN_PATH / "split_audit.json").read_text())["audit"]
display(
    {
        "recipe_sha256": pointer["recipe_sha256"],
        "environment": run_metadata["recipe"]["environment"],
        "hardware": run_metadata["hardware"],
        "free_disk_gib": run_metadata["free_disk_gib"],
    }
)
display(audit["partition_counts"], audit["fold_counts"], audit["cv_sample_fold_counts"])
display(
    {
        key: audit[key]
        for key in ["duplicate_groups", "known_group_partition_overlap", "known_group_fold_overlap"]
    },
    audit["distribution_deviations"],
)

## 03 · Controlled CV and selected-model holdout evaluation

The original scientific recipe is unchanged: word (1,2)-grams, character (3,5)-grams, 40k/60k feature caps, min_df 3, sublinear TF, float32 sparse blocks and preserved case. Character analysis collapses repeated whitespace internally. All four configurations use C = 1.0; additional regularization settings are deferred. Language/domain are never predictive features.

Completed CV folds persist score, OOF predictions and checksums immediately. A compatible rerun skips them; an interrupted active fit restarts that fold. Completed holdout evaluation is restored. Only the highest mean three-class CV macro-F1 candidate is evaluated on the holdout. Unknown source-family leakage remains unresolved.

If this execution finishes, preserve its outputs with Kaggle's Save Version workflow. Stop interactive training before launching another saved run. Session storage is temporary: completed files are durable only after they have been saved or downloaded.

In [ ]:
if RUN_TRAINING and not (RUN_FINAL_SUBMISSION and FINAL_STAGE == "inference"):
    run_stage("cv-holdout")
else:
    display({"training_status": "explicitly disabled"})

comparison_path = RUN_PATH / "model_comparison.csv"
if comparison_path.exists():
    with comparison_path.open(newline="") as stream:
        comparison = list(csv.DictReader(stream))
    headers = list(comparison[0])
    table = (
        "<table><thead><tr>"
        + "".join(f"<th>{html.escape(name)}</th>" for name in headers)
        + "</tr></thead><tbody>"
    )
    table += "".join(
        "<tr>" + "".join(f"<td>{html.escape(row[name])}</td>" for name in headers) + "</tr>"
        for row in comparison
    )
    display(HTML(table + "</tbody></table>"))
    metrics = json.loads((RUN_PATH / "holdout_metrics.json").read_text())
    display(
        {
            name: metrics[name]
            for name in ["macro_f1", "weighted_f1", "accuracy", "model_class_order"]
        },
        metrics["per_class"],
    )
    display(FileLink(str(comparison_path)))
else:
    display({"model_results": "not measured yet"})

## 04 · Optional final refit or inference-only export

Leave this disabled until CV/holdout selection is complete. With `FINAL_STAGE = "submission"`, a missing final checkpoint is fitted on all 272,352 labeled records, then every test row is predicted in batches. Compatible completed checkpoints are restored without refitting. `FINAL_STAGE = "inference"` requires an existing final checkpoint and never starts training.

Output validation enforces 25,000 unique IDs, sample order, exactly `id,label` and the three lowercase labels. Model, CSV and input fingerprints accompany the export. No automatic submission occurs.

For inference-only Run All, also set `RUN_TRAINING = False`. The training cell additionally skips fitting automatically when final inference-only mode is selected.

The final refit uses streamed vocabulary/document frequencies and chunked sparse transformation to avoid allocating the unpruned full-corpus count matrices. Equivalence to the original sklearn vocabulary, IDF and predictions is covered by a focused test. This preserves the selected C, n-grams, feature caps, normalization and training population.

In [ ]:
if RUN_FINAL_SUBMISSION:
    if FINAL_STAGE not in {"submission", "inference"}:
        raise ValueError("Choose submission or inference for FINAL_STAGE")
    command = [
        str(RUNTIME_PYTHON),
        "-I",
        str(CODE_ROOT / "scripts/run_bounded_submission.py"),
        "--code-root",
        str(CODE_ROOT),
        "--run-path",
        str(RUN_PATH),
        "--input-root",
        str(INPUT_ROOT),
    ]
    if FINAL_STAGE == "inference":
        command.append("--inference-only")
    with subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as process:
        if process.stdout is None:
            raise RuntimeError("Final export output pipe is unavailable")
        for line in process.stdout:
            logger.info("%s", line.rstrip())
        return_code = process.wait()
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)
    for path in sorted((RUN_PATH / "submissions").glob("*.csv")):
        display(FileLink(str(path)))
else:
    display({"final_export_status": "explicitly disabled"})

## 05 · Save and resume

Save this private notebook's output version or download `/kaggle/working/hm3_baseline/`. To resume in a fresh session, attach the previous saved notebook outputs as another input together with the same code bundle, frozen split and competition data. The runner restores the exact matching recipe directory, verifies its completion records, skips completed folds and reuses completed holdout/final-model outputs.

Do not edit settings merely to resume an interrupted run. A source, split, code, feature or library-version mismatch creates a different recipe or fails compatibility validation. No promise is made about Kaggle session duration, empirical peak RAM, model convergence or completion before the first real hosted run.